# Organism_Hierarchy — where NCI's hierarchy places the MICROORG terms

Reads the NCIt hierarchy above every term of the SDTM CT codelist MICROORG (C85491,
Microorganism) and reports where each term sits: the NCIt paths up to the root, the
NCIt branch, and whether a family-rank concept is on a path.

Question it answers: can a consumer carry an organism by its family instead of the
whole codelist? NCIt publishes no taxonomic rank, so "family" is read here from the
names, by the suffix rule of the nomenclature codes. **That rule is ours**, not
NCI's; everything else is NCI's hierarchy as published in the FLAT file.

It reports only. Nothing in the Findings files changes and nothing reads from this
output.

**Inputs**
- `sdtm-test-codes/downloads/SDTM_Terminology.txt` — the MICROORG codelist and its terms
- `sdtm-test-codes/downloads/Thesaurus.txt` — NCIt FLAT (parents, preferred names); gitignored, filled by the CT release run
- `sdtm-test-codes/downloads/Thesaurus.FLAT.zip` — for the FLAT build date
- `Reference_Versions.json` — the pins

**Output** — `domain-behaviour/interim/Organism_Hierarchy.xlsx`

| Sheet | Grain | Content |
|---|---|---|
| `ReadMe` | — | Purpose, scope, versions, known gaps, sheets, the family rule |
| `Terms` | MICROORG term | NCIt paths to the root, branch, family-rank concept on a path (ours) |
| `By_Branch` | NCIt branch (two levels below the root) | Terms, with and without a family-rank concept |
| `Families` | family-rank concept | Terms under it, branch |
| `Pins` | — | Input versions |

Column headers: green = SDTM CT or NCIt as published; blue = our reading (the family rule).


## 1. Setup


In [1]:
import json
import re
import zipfile
from datetime import date
from pathlib import Path

import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter

REPO = Path.cwd().parent.parent  # notebooks/ -> domain-behaviour/ -> repo root
VERSIONS = json.loads((REPO / 'Reference_Versions.json').read_text(encoding='utf-8'))
CT_FILE = REPO / 'sdtm-test-codes' / 'downloads' / 'SDTM_Terminology.txt'
FLAT_ZIP = REPO / 'sdtm-test-codes' / 'downloads' / 'Thesaurus.FLAT.zip'
FLAT_FILE = REPO / 'sdtm-test-codes' / 'downloads' / 'Thesaurus.txt'
OUT = REPO / 'domain-behaviour' / 'interim' / 'Organism_Hierarchy.xlsx'

CODELIST = 'MICROORG'

for path in [CT_FILE, FLAT_ZIP, FLAT_FILE]:
    if not path.exists():
        raise FileNotFoundError(f'{path} not found: run the CT release run (sdtm-test-codes) first')
    print(f'  {path.relative_to(REPO)}')

with zipfile.ZipFile(FLAT_ZIP) as zf:
    _info = zf.getinfo('Thesaurus.txt')
FLAT_DATE = date(*_info.date_time[:3]).isoformat()
if _info.file_size != FLAT_FILE.stat().st_size:
    raise RuntimeError(f'{FLAT_FILE.name} ({FLAT_FILE.stat().st_size:,} bytes) is not the file in '
                       f'{FLAT_ZIP.name} ({_info.file_size:,} bytes): unzip again')
if FLAT_DATE != VERSIONS['NCIt']:
    raise RuntimeError(f"NCIt pin {VERSIONS['NCIt']} != FLAT build date {FLAT_DATE}: update Reference_Versions.json")
print(f"NCIt FLAT build date: {FLAT_DATE} (= pin); SDTM CT pin: {VERSIONS['SDTM_CT']}")


  sdtm-test-codes/downloads/SDTM_Terminology.txt
  sdtm-test-codes/downloads/Thesaurus.FLAT.zip
  sdtm-test-codes/downloads/Thesaurus.txt
NCIt FLAT build date: 2026-09-29 (= pin); SDTM CT pin: 2026-09-25


## 2. Load

The MICROORG codelist is found by its submission value; its terms are the CT rows with
that codelist code. Every term code must be in the FLAT file. The NCIt preferred name of
a concept is the first entry of the FLAT synonyms column, as in `SDTM_CT_Term_NCIt_Synonyms`.


In [2]:
ct = pd.read_csv(CT_FILE, sep='\t', dtype=str).fillna('')
_cl = ct[(ct['Codelist Code'] == '') & (ct['CDISC Submission Value'] == CODELIST)]
if len(_cl) != 1:
    raise RuntimeError(f'{CODELIST} found {len(_cl)} times as a codelist in {CT_FILE.name}')
CODELIST_CODE = _cl['Code'].iloc[0]
CODELIST_NAME = _cl['Codelist Name'].iloc[0]
terms = ct.loc[ct['Codelist Code'] == CODELIST_CODE,
               ['Code', 'CDISC Submission Value', 'NCI Preferred Term']].copy()
terms.columns = ['NCIt_Code', 'Submission_Value', 'NCI_Preferred_Term']
terms = terms.sort_values('Submission_Value').reset_index(drop=True)
if terms['NCIt_Code'].duplicated().any():
    raise RuntimeError(f"Duplicate term codes in {CODELIST}: {terms.loc[terms['NCIt_Code'].duplicated(), 'NCIt_Code'].tolist()}")
print(f'{CODELIST} ({CODELIST_CODE}, {CODELIST_NAME}): {len(terms):,} terms')

FLAT_COLS = ['code', 'url', 'parents', 'synonyms', 'definition',
             'display_name', 'concept_status', 'semantic_type', 'subset_membership']
flat = pd.read_csv(FLAT_FILE, sep='\t', header=None, names=FLAT_COLS, dtype=str, na_filter=False)
parents_of = {c: [p for p in v.split('|') if p] for c, v in zip(flat['code'], flat['parents'])}
pref_name = {c: v.split('|')[0].strip() for c, v in zip(flat['code'], flat['synonyms'])}

_missing = sorted(set(terms['NCIt_Code']) - set(parents_of))
if _missing:
    raise RuntimeError(f'{len(_missing)} {CODELIST} term codes not in the FLAT file, e.g. {_missing[:10]}')
print(f'FLAT concepts: {len(flat):,}')


MICROORG (C85491, Microorganism): 1,707 terms
FLAT concepts: 213,524


## 3. NCIt paths

Every path from the term up to a root (a concept without parents), following all
parents. A path is written from the root down to the term's direct parent; several
paths are separated by `|`. The branch of a path is its first two concepts below the
root, the term itself included; a term one level below the root has an empty second level.


In [3]:
def paths_up(code):
    """All parent chains from code to a root, each as a list from the direct parent upwards."""
    ps = parents_of[code]
    if not ps:
        return [[]]
    out = []
    for p in ps:
        if p not in parents_of:
            raise RuntimeError(f'{code}: parent {p} has no FLAT row')
        for rest in paths_up(p):
            out.append([p] + rest)
    return out


term_paths = {c: [list(reversed(p)) for p in paths_up(c)] for c in terms['NCIt_Code']}
_no_parent = [c for c, ps in term_paths.items() if ps == [[]]]
if _no_parent:
    raise RuntimeError(f'{CODELIST} terms without NCIt parents: {_no_parent}')
_at_root = [c for c, ps in term_paths.items() if any(len(p) < 1 for p in ps)]
if _at_root:
    raise RuntimeError(f'{CODELIST} terms that are a root: {_at_root}')
roots = sorted({p[0] for ps in term_paths.values() for p in ps})
print(f'Roots reached: {[(r, pref_name[r]) for r in roots]}')
print(f'Terms with more than one path: {sum(len(ps) > 1 for ps in term_paths.values()):,}')


Roots reached: [('C14250', 'Organism')]
Terms with more than one path: 64


## 4. Family-rank concepts (our reading)

NCIt has no rank property. A concept is read as family rank when its NCIt preferred
name is one capitalised word ending in a family suffix of the nomenclature codes:
`-aceae` (prokaryotes, fungi, algae, plants), `-viridae` (viruses), `-idae` (animals,
incl. protozoa named under the zoological code). The term itself counts when it is a
family. The run stops if a term reaches more than one family-rank concept: the rule
or the hierarchy would then need a look.


In [4]:
FAMILY_NAME = re.compile(r'^[A-Z][a-z]+(aceae|viridae|idae)$')


def families_of(code):
    on_path = {code} | {c for p in term_paths[code] for c in p}
    return sorted(c for c in on_path if FAMILY_NAME.match(pref_name[c]))


term_families = {c: families_of(c) for c in terms['NCIt_Code']}
_multi = {c: f for c, f in term_families.items() if len(f) > 1}
if _multi:
    raise RuntimeError(f'Terms reaching more than one family-rank concept: {_multi}')
print(f'Terms with a family-rank concept: {sum(bool(f) for f in term_families.values()):,} of {len(terms):,}')


Terms with a family-rank concept: 1,222 of 1,707


## 5. Sheets


In [5]:
def branch(p, code):
    chain = p + [code]  # root ... direct parent, term
    return (chain[1], chain[2] if len(chain) > 2 else '')


rows = []
for t in terms.itertuples(index=False):
    ps = term_paths[t.NCIt_Code]
    brs = sorted({branch(p, t.NCIt_Code) for p in ps})
    fam = term_families[t.NCIt_Code]
    rows.append({
        'NCIt_Code': t.NCIt_Code,
        'Submission_Value': t.Submission_Value,
        'NCI_Preferred_Term': t.NCI_Preferred_Term,
        'NCIt_Parent_Codes': '|'.join(parents_of[t.NCIt_Code]),
        'NCIt_Parent_Terms': '|'.join(pref_name[p] for p in parents_of[t.NCIt_Code]),
        'Path_Count': len(ps),
        'NCIt_Paths': '|'.join(' > '.join(pref_name[c] for c in p) for p in ps),
        'NCIt_Path_Codes': '|'.join(' > '.join(p) for p in ps),
        'Branch_Codes': '|'.join(f'{a} > {b}' for a, b in brs),
        'Branch_Terms': '|'.join(f"{pref_name[a]} > {pref_name.get(b, '')}" for a, b in brs),
        'Family_Code': fam[0] if fam else '',
        'Family_Term': pref_name[fam[0]] if fam else '',
        'Term_Is_Family': 'Yes' if fam and fam[0] == t.NCIt_Code else ('No' if fam else ''),
    })
terms_sheet = pd.DataFrame(rows)

_br = terms_sheet.assign(Branch=terms_sheet['Branch_Codes'].str.split('|')).explode('Branch')
_br[['Branch_1_Code', 'Branch_2_Code']] = _br['Branch'].str.split(' > ', expand=True)
by_branch = (_br.groupby(['Branch_1_Code', 'Branch_2_Code'])
             .agg(Terms=('NCIt_Code', 'size'),
                  With_Family=('Family_Code', lambda s: int((s != '').sum())),
                  Families=('Family_Code', lambda s: s[s != ''].nunique()))
             .reset_index())
by_branch['Without_Family'] = by_branch['Terms'] - by_branch['With_Family']
by_branch.insert(1, 'Branch_1_Term', by_branch['Branch_1_Code'].map(pref_name))
by_branch.insert(3, 'Branch_2_Term', by_branch['Branch_2_Code'].map(lambda c: pref_name[c] if c else ''))
by_branch = by_branch[['Branch_1_Code', 'Branch_1_Term', 'Branch_2_Code', 'Branch_2_Term',
                       'Terms', 'With_Family', 'Without_Family', 'Families']]
by_branch = by_branch.sort_values(['Branch_1_Term', 'Branch_2_Term']).reset_index(drop=True)

_f = terms_sheet[terms_sheet['Family_Code'] != '']
families = (_f.groupby(['Family_Code', 'Family_Term'])
            .agg(Terms=('NCIt_Code', 'size'),
                 Term_Is_Family=('Term_Is_Family', lambda s: int((s == 'Yes').sum())),
                 Branch_Terms=('Branch_Terms', lambda s: '|'.join(sorted({b for v in s for b in v.split('|')}))))
            .reset_index().sort_values(['Terms', 'Family_Term'], ascending=[False, True]).reset_index(drop=True))

_multi_branch = int((terms_sheet['Branch_Codes'].str.count(r'\|') > 0).sum())
print(f'Terms: {len(terms_sheet):,}  branches: {len(by_branch)}  families: {len(families)}')
print(f'Terms in more than one branch (counted in each in By_Branch): {_multi_branch}')


Terms: 1,707  branches: 19  families: 93
Terms in more than one branch (counted in each in By_Branch): 46


## 6. Write


In [6]:
n_terms = len(terms_sheet)
n_fam = int((terms_sheet['Family_Code'] != '').sum())
_b1 = (terms_sheet.assign(B1=terms_sheet['Branch_Terms'].str.split('|')).explode('B1')
       .assign(B1=lambda d: d['B1'].str.split(' > ').str[0])
       .drop_duplicates(['NCIt_Code', 'B1']))
_b1 = _b1.groupby('B1').agg(Terms=('NCIt_Code', 'size'),
                            With_Family=('Family_Code', lambda s: int((s != '').sum())))

readme = [
    ('Organism_Hierarchy — NCIt hierarchy above the MICROORG terms', 'title'),
    ('', None),
    ('PURPOSE', 'section'),
    ('Where NCI\'s hierarchy places each term of the SDTM CT codelist MICROORG, and whether a', None),
    ('family-rank concept is on its NCIt path, so a consumer can see where an organism can be', None),
    ('carried by its family and where only by NCI\'s own grouping. Report only.', None),
    ('', None),
    ('SCOPE', 'section'),
    (f'{CODELIST} ({CODELIST_CODE}, {CODELIST_NAME}): {n_terms:,} terms. All NCIt parents are followed to a root.', None),
    ('', None),
    ('SOURCE VERSIONS', 'section'),
    (f"SDTM CT {VERSIONS['SDTM_CT']} (NCI EVS); NCIt FLAT {FLAT_DATE}.", None),
    ('', None),
    ('KNOWN GAPS', 'section'),
    ('NCIt publishes no taxonomic rank. Family_Code / Family_Term (blue) are our reading: a concept', None),
    ('whose NCIt preferred name is one capitalised word ending in -aceae, -viridae or -idae (the family', None),
    ('suffixes of the nomenclature codes). A family named otherwise is not found; a non-family name with', None),
    ('such an ending would be read as one.', None),
    (f'{n_fam:,} of {n_terms:,} terms have a family-rank concept on a path; none has more than one (the run', None),
    ('stops if one does). By first branch below the root:', None),
]
for b1, r in _b1.iterrows():
    readme.append((f'    {b1}: {r.Terms:,} terms, {r.With_Family:,} with a family-rank concept', None))
readme += [
    ('Branches are NCI\'s as published. Some are groupings rather than taxonomy (e.g. under "Other Organism', None),
    ('Groupings"); they are listed as NCI shows them. A term in more than one branch is counted in each', None),
    (f'in By_Branch ({_multi_branch} terms).', None),
    ('', None),
    ('SHEETS', 'section'),
    ('Terms — one row per MICROORG term: NCIt parents, all paths to the root (root first, | between', None),
    ('    paths), branches (first two concepts below the root, the term included), family-rank concept (ours).', None),
    ('By_Branch — one row per branch: terms, with and without a family-rank concept, distinct families.', None),
    ('Families — one row per family-rank concept: terms under it, how many are the family itself, branches.', None),
    ('Pins — input versions.', None),
    ('', None),
    ('COLUMN COLOURS', 'section'),
    ('Green: SDTM CT or NCIt as published. Blue: our reading (the family rule).', None),
    ('', None),
    ('Built by domain-behaviour/notebooks/Organism_Hierarchy.ipynb.', None),
]

pins = pd.DataFrame([
    ('SDTM_CT', VERSIONS['SDTM_CT']),
    ('NCIt_FLAT', FLAT_DATE),
    ('Codelist', f'{CODELIST} ({CODELIST_CODE})'),
    ('Terms', n_terms),
], columns=['Key', 'Value'])

GREEN = PatternFill('solid', fgColor='C6EFCE')
BLUE = PatternFill('solid', fgColor='4472C4')
HEADER_FONT = Font(name='Arial', size=10, bold=True)
BLUE_FONT = Font(name='Arial', size=10, bold=True, color='FFFFFF')
BODY_FONT = Font(name='Arial', size=10)
OURS = {'Family_Code', 'Family_Term', 'Term_Is_Family', 'With_Family', 'Without_Family', 'Families'}


def write_sheet(ws, df, widths):
    ws.append(list(df.columns))
    for i, col in enumerate(df.columns, start=1):
        cell = ws.cell(row=1, column=i)
        cell.fill = BLUE if col in OURS else GREEN
        cell.font = BLUE_FONT if col in OURS else HEADER_FONT
        ws.column_dimensions[get_column_letter(i)].width = widths.get(col, 14)
    for row in df.itertuples(index=False):
        ws.append(list(row))
    ws.freeze_panes = 'A2'


wb = Workbook()
ws = wb.active
ws.title = 'ReadMe'
for text, style in readme:
    ws.append([text])
    ws.cell(row=ws.max_row, column=1).font = (Font(name='Arial', size=12, bold=True) if style == 'title'
                                              else HEADER_FONT if style == 'section' else BODY_FONT)
ws.column_dimensions['A'].width = 110

write_sheet(wb.create_sheet('Terms'), terms_sheet, {
    'NCIt_Code': 12, 'Submission_Value': 36, 'NCI_Preferred_Term': 36,
    'NCIt_Parent_Codes': 18, 'NCIt_Parent_Terms': 36, 'Path_Count': 10,
    'NCIt_Paths': 90, 'NCIt_Path_Codes': 60, 'Branch_Codes': 22, 'Branch_Terms': 40,
    'Family_Code': 12, 'Family_Term': 24, 'Term_Is_Family': 12})
write_sheet(wb.create_sheet('By_Branch'), by_branch, {
    'Branch_1_Code': 14, 'Branch_1_Term': 28, 'Branch_2_Code': 14, 'Branch_2_Term': 36})
write_sheet(wb.create_sheet('Families'), families, {
    'Family_Code': 12, 'Family_Term': 28, 'Branch_Terms': 60})
write_sheet(wb.create_sheet('Pins'), pins, {'Key': 14, 'Value': 24})

OUT.parent.mkdir(parents=True, exist_ok=True)
wb.save(OUT)
print(f'Wrote {OUT.relative_to(REPO)}')


Wrote domain-behaviour/interim/Organism_Hierarchy.xlsx


## 7. Summary


In [7]:
print(_b1.to_string())
print()
print(by_branch.sort_values('Terms', ascending=False).head(20).to_string(index=False))


                          Terms  With_Family
B1                                          
Bacteria                   1168         1120
Eukaryota                   317            0
Other Organism Groupings     43            1
Virus                       183          102

Branch_1_Code            Branch_1_Term Branch_2_Code            Branch_2_Term  Terms  With_Family  Without_Family  Families
       C14187                 Bacteria        C76196           Proteobacteria    497          491               6        30
       C14187                 Bacteria        C76197               Firmicutes    345          311              34        19
       C14187                 Bacteria        C76195           Actinobacteria    198          196               2        20
       C25796                Eukaryota        C14209                   Fungus    183            0             183         0
       C14283                    Virus        C14269                RNA Virus    118          101            